# Go2 energy-efficient walking: training on a Kaggle GPU (MJX + Brax PPO)

**Before running:**
1. Right panel → *Session options* → **Accelerator: GPU T4 x2**, **Internet: On**
   (needs a phone-verified Kaggle account).
2. *Add Input* → your private dataset **go2-bundle** (made with `kaggle\make_bundle.ps1` on your PC).
3. Set `BATCH` in the *Train* cell (section 4).

To run unattended (up to 12 h, browser can be closed): **Save Version → Save & Run All (Commit)**.
Results then appear in the version's **Output** tab (`runs.zip`).

## 1. Copy the project from the dataset

In [ ]:
import glob, os, shutil, zipfile
found = glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)
if found:  # Kaggle unzipped the bundle (usual case)
    shutil.copytree(os.path.dirname(found[0]), "/kaggle/working/go2", dirs_exist_ok=True)
else:      # bundle kept as a .zip
    zips = glob.glob("/kaggle/input/**/go2_bundle*.zip", recursive=True)
    assert zips, "go2-bundle dataset not found: use 'Add Input' to attach it"
    zipfile.ZipFile(zips[0]).extractall("/kaggle/working/go2")
os.chdir("/kaggle/working/go2")
print(sorted(os.listdir(".")))

## 2. Install (versions tested on the PC; see pyproject.toml)

In [ ]:
# brax 0.14.2 needs jax.device_put_replicated, removed in jax 0.10 -> jax 0.9.2 (CUDA build)
!pip install -q "jax[cuda12]==0.9.2" -e ".[mjx]"

In [ ]:
os.environ["XLA_FLAGS"] = "--xla_gpu_triton_gemm_any=True"  # faster matmuls on GPU (Playground tip)
# Check the GPU in a separate process: if this notebook imported JAX itself, it would
# reserve ~75 % of GPU memory and leave too little for the training process.
!python -c "import jax; print(jax.__version__, jax.devices()); assert jax.devices()[0].platform == 'gpu', 'No GPU: set Accelerator to GPU T4 x2'"

## 3. Checks (optional, ~2 min): MJX env vs CPU env, then GPU speed

In [ ]:
!python scripts/check_mjx.py --foot rigid

In [ ]:
!python scripts/bench_mjx.py --envs 4096

## 4. Train one batch of the energy study (rigid foot)
All runs fine-tune `rigid_base` (bundled as `init/rigid_base`, made with `make_bundle.ps1 -Init rigid_base`)
for 50M steps (~1.8 h each) with one energy term:
- `phase`: the paper's energy-phase term (Eq. 15); on the rigid foot it penalises changes in leg power.
- `power`: mechanical power, sum of |torque x joint speed|.
- `torque`: sum of torque².

Weights of different types are matched by `scripts/calibrate_eng.py` (same mean penalty on `rigid_base`):
phase 3e-4 / 1e-3 / 3e-3 = power 1e-3 / 3.4e-3 / 1e-2 = torque 1.6e-4 / 5.5e-4 / 1.6e-3.
Already done (stage 1): `rigid_ft0` (no energy term), `rigid_ft_eng` (phase 3e-3), `rigid_ft_eng_lo` (phase 3e-4).

Set `BATCH`, then Save & Run All (Commit). One batch per commit; A and B fit in the 12 h limit.
If a run collapses for good (episode length stays low), rerun it with `LR = 1e-4`.

In [ ]:
BATCH = "A"     # "A", then "B"
LR = 3e-4
BATCHES = {
    # name suffix: (energy type, weight)
    "A": {"phase_1e-4": ("phase", 1e-4), "phase_1e-3": ("phase", 1e-3), "phase_1e-2": ("phase", 1e-2),
          "power_1e-3": ("power", 1e-3), "power_3.4e-3": ("power", 3.4e-3)},
    "B": {"power_1e-2": ("power", 1e-2), "torque_1.6e-4": ("torque", 1.6e-4),
          "torque_5.5e-4": ("torque", 5.5e-4), "torque_1.6e-3": ("torque", 1.6e-3)},
}
OUT = "/kaggle/working/runs"
for suffix, (etype, w) in BATCHES[BATCH].items():
    name = f"rigid_{suffix}"
    print("=" * 30, name, "=" * 30, flush=True)
    !python scripts/train_mjx.py --name {name} --foot rigid --eng-type {etype} --eng-weight {w} --steps 50_000_000 --init-from rigid_base --lr {LR} --out {OUT}

## 5. Package results (download `runs.zip` from the Output tab, unzip into `C:\mujoco_go2\`)

In [ ]:
os.chdir("/kaggle/working")
shutil.rmtree("go2", ignore_errors=True)  # keep the output small: only runs/ is needed
!zip -qr runs.zip runs && ls -la runs.zip runs/*